# Getting Started with Automated-LLM-Probes

In [1]:
from __future__ import annotations
import os,time,tqdm,pandas as pd
from pathlib import Path
import automated_intelligence_tests as ait
import automated_llm_probes as alp
assert os.environ.get("OPENAI_API_KEY")

### Models available

In [2]:
models = alp.load_models()
print(f"|Models available: {len(models)}|\n")
for m in models:
    print(f"  {m['name'][:20]:20s} {m['vendor'][:11]:12s} \
{m['api'][:11]:12s} {m['model_id'][:12]:14s} {m['status'][:12]:12s}")

|Models available: 90|

  grok-code-fast       xai          spacexai     grok-code-fa   ok          
  grok-2               xai          spacexai     grok-2         failed      
  grok-4               xai          spacexai     grok-4         ok          
  grok-4.2             xai          spacexai     grok-4.20-03   ok          
  grok-4.2-reasoning   xai          spacexai     grok-4.20-03   ok          
  grok-4.3             xai          spacexai     grok-4.3       ok          
  grok-4.5             xai          spacexai     grok-4.5       ok          
  grok-build-0.1       xai          spacexai     grok-build-0   ok          
  grok-4.6             xai          spacexai     grok-4.6       ok          
  grok-4.2-multi-agent xai          spacexai     grok-4.20-mu   failed      
  hunyuan-3            tencent      hunyuan      hy3            ok          
  hunyuan-lite         tencent      hunyuan      hunyuan-lite   failed      
  qwen-turbo           qwen         qwen         qwe

### Probe models

In [3]:
def probe_models(model_rows,prompt=[{"role":"user","content":"reply ok"}]):
    retries = alp.MAX_RETRIES; alp.MAX_RETRIES = 1
    try:
        for model in tqdm.tqdm(model_rows):
            start = time.time()
            try:
                model['reply'] = alp.call_model(model,prompt)
                model["status"] = "ok" if model['reply'].strip() else "failed"
                model["errors"] = None
            except Exception as e:
                model['reply'] = None
                model["status"] = 'failed'
                model["errors"] = str(e).lower()
            model["speed_per_call"] = f"{round((time.time()-start),2)} seconds"
    finally:
        alp.MAX_RETRIES = retries
    return model_rows

models = alp.load_models()
probes = probe_models(models)
probes = pd.DataFrame(probes).set_index('name')

100%|████████████████████████████████████████████████████████████████████| 91/91 [04:51<00:00,  3.21s/it]


In [4]:
probes.to_csv('./models.csv')
probes.head()

,vendor,api,model_id,release_date,temperature,speed_per_call,status,reply,errors
name,,,,,,,,,
grok-code-fast,xai,spacexai,grok-code-fast,26-Aug-25,0.5,9.83 seconds,ok,Ok.,None
grok-2,xai,spacexai,grok-2,13-Aug-24,default,2.19 seconds,failed,None,grok-2 failed after 1 tries: error code: 400 -...
grok-4,xai,spacexai,grok-4,9-Jul-25,0.5,1.95 seconds,ok,ok,None
grok-4.2,xai,spacexai,grok-4.20-0309-non-reasoning,9-Mar-26,0.5,0.69 seconds,ok,ok,None
grok-4.2-reasoning,xai,spacexai,grok-4.20-0309-reasoning,9-Mar-26,0.5,1.95 seconds,ok,ok,None


### Probed tasks

In [5]:
path = Path('./data/'); n=1
print(f"|Probed tasks: {len([p for p in path.iterdir() if p.is_dir()])}|\n")
for d in path.iterdir():
    if (d.is_dir()) and ('.' not in d.name):
        count = sum(1 for f in d.rglob('*') if f.is_file())
        print(f"  {n}. {d.name.upper()[:7]:8}:  {count}"); n+=1

|Probed tasks: 4|

  1. AUT     :  16574
  2. CWT     :  22194
  3. DAT     :  10781


### Tests availabe

In [6]:
ait_counts = ait.list_available_tests()
print(f'|Available tests: {len(ait_counts)}|\n')
for i,v in ait_counts.items():
    print(f'  {i} : {v}')

|Available tests: 4|

  AUT : Alternative Uses Task
  CAT : Convergent Association Task
  CWT : Creative Writing Task
  DAT : Divergent Association Task


### Smoke-run

{'name': 'grok-code-fast',
 'vendor': 'xai',
 'api': 'spacexai',
 'model_id': 'grok-code-fast',
 'release_date': '26-Aug-25',
 'temperature': '0.5',
 'speed_per_call': '9.83 seconds',
 'status': 'ok',
 'reply': 'Ok.',
 'errors': ''}

In [22]:
from pathlib import Path
import pickle
import automated_llm_probes as alp

ROOT = Path("./data")
before = {p.resolve() for p in ROOT.rglob("*.pickle")} if ROOT.exists() else set()
ready = alp.ready_models()

alp.collect(
    "DAT",
    models=[ready[0]],
    n_per_model=1,
    n_to_topup=True,
    scoring=True,
    seed=0,)

after = {p.resolve() for p in ROOT.rglob("*.pickle")}
new = sorted(after - before)
print(f"\nnew pickles: {len(new)}")
if not new:
    raise SystemExit("collect wrote nothing — check API error prints above")
    
for p in new:
    row = pickle.load(p.open("rb"))
    print("\n===", p, "===")
    print(row)

for p in new:
    p.unlink()
    print("deleted", p)

left = {p.resolve() for p in ROOT.rglob("*.pickle")} if ROOT.exists() else set()
assert left == before, "smoke delete touched pre-existing pickles"
print("\nsmoke ok — pre-existing pickle count unchanged:", len(before))

  grok-code-fast: 5 collected, 1 to collect


DAT: 100%|█████████████████████████████████████████████████████████████████| 1/1 [00:15<00:00, 15.82s/it]



new pickles: 1

=== /Users/daweiwang/Library/CloudStorage/Dropbox-Personal/GitHub_Repositories/Automated-LLM-Probes/Automated-LLM-Probes/data/dat/grok-code-fast/0.5/5996571cb9877090.pickle ===
{'task': 'DAT', 'model_name': 'grok-code-fast', 'model_id': 'grok-code-fast', 'provider': 'spacexai', 'rep': 5, 'temperature': '0.5', 'kwargs': {'test': 'dat', 'cue': None, 'n_words': 10, 'instructions': 'Please enter 10 words that are as different from each other as possible, in all meanings and uses of the words.\n\nRules:\nOnly single words in English.\nOnly nouns (things, objects, concepts).\nNo proper nouns (no specific people or places).\nNo specialised vocabulary or technical terms.\nThink of the words on your own.\n\nNotes:\nReturn words as comma-separated list. Do not return anything else.', 'response_format': {'word_1': '...', 'word_2': '...', 'word_3': '...', 'word_4': '...', 'word_5': '...', 'word_6': '...', 'word_7': '...', 'word_8': '...', 'word_9': '...', 'word_10': '...'}}, 'prom

### Trial-run

In [7]:
models_to_try = [
    'claude-haiku-4.5',
    'claude-opus-4.5',
    'claude-opus-4.7',
    'claude-opus-5',
    'claude-sonnet-4.5',
    'gpt-3.5-turbo',
    'gpt-4-turbo',
    'gpt-4-turbo',
    'gpt-4o',
    'gpt-4o-mini',
    'gpt-5.4',
    'grok-4.2',
    'grok-4.3',
    'grok-4.5',
    'grok-build-0.1',
    'llama-3.1-8b',
    'llama-3.2-3b',
    'llama-4-guard-12b',
    'llama-4-maverick',
    'llama-4-scout']

cues_to_try = ["brick", "paperclip"]

models = alp.ready_models()
models_to_try = [m for m in models if m["name"] in models_to_try]
sorted([m["name"] for m in models_to_try])

['claude-haiku-4.5',
 'claude-opus-4.5',
 'claude-opus-4.7',
 'claude-opus-5',
 'claude-sonnet-4.5',
 'gpt-3.5-turbo',
 'gpt-4-turbo',
 'gpt-4-turbo',
 'gpt-4o',
 'gpt-4o-mini',
 'gpt-5.4',
 'grok-4.2',
 'grok-4.3',
 'grok-4.5',
 'grok-build-0.1',
 'llama-3.1-8b',
 'llama-3.2-3b',
 'llama-4-guard-12b',
 'llama-4-maverick',
 'llama-4-scout']

In [8]:
alp.collect("AUT", models=models_to_try, n_per_model=0)

  grok-4.2: 832/0 done — skip
  grok-4.3: 825/0 done — skip
  grok-4.5: 828/0 done — skip
  grok-build-0.1: 620/0 done — skip
  gpt-3.5-turbo: 921/0 done — skip
  gpt-4-turbo: 825/0 done — skip
  gpt-4o: 870/0 done — skip
  gpt-5.4: 861/0 done — skip
  gpt-4o-mini: 829/0 done — skip
  gpt-4-turbo: 825/0 done — skip
  llama-4-guard-12b: 600/0 done — skip
  llama-4-scout: 620/0 done — skip
  llama-4-maverick: 600/0 done — skip
  llama-3.2-3b: 600/0 done — skip
  llama-3.1-8b: 600/0 done — skip
  claude-sonnet-4.5: 827/0 done — skip
  claude-haiku-4.5: 830/0 done — skip
  claude-opus-4.5: 829/0 done — skip
  claude-opus-4.7: 859/0 done — skip
  claude-opus-5: 855/0 done — skip


### Parse & score responses

In [8]:
from automated_llm_probes import _parse_and_score

p, s = _parse_and_score("dat", "arm, eyes, feet, hand, head, leg, body", {})
assert isinstance(p, list) and len(p) >= 7
assert s is None or isinstance(s, float)
print(p,s)

p, s = _parse_and_score("aut", "doorstop\npaperweight", {"cue": "brick"})
assert isinstance(p, dict) and p["cue"] == "brick"
assert s is None or isinstance(s, float)
print(p,s)

p, s = _parse_and_score("cat", "lake", {})
assert p is None and s is None
print(p,s)

['arm', 'eyes', 'feet', 'hand', 'head', 'leg', 'body'] 50.30972943419501
{'cue': 'brick', 'responses': ['doorstop', 'paperweight']} 0.8485623937100173
None None


### Load & merge data

In [9]:
def load_task(df, task):
    task = task.lower()
    df = df.copy()
    if "cue" not in df.columns:
        df["cue"] = df.get("kwargs", pd.Series(dtype=object)).map(
            lambda k: (k or {}).get("cue") if isinstance(k, dict) else None
        )

    def _clean(parsed):
        if parsed is None:
            return None
        if task == "dat" and isinstance(parsed, list):
            return parsed
        if task == "aut" and isinstance(parsed, dict):
            return parsed.get("responses")
        if task == "cwt" and isinstance(parsed, dict):
            return parsed.get("story")
        return parsed

    if "parsed" not in df.columns:
        df["parsed"] = None
    df["response_clean"] = df["parsed"].map(_clean)

    if task == "dat":
        nouns = df["response_clean"].apply(
            lambda xs: list(xs) + [None] * 10 if isinstance(xs, list) else [None] * 10
        )
        for i in range(10):
            df[f"noun_{i}"] = nouns.map(lambda xs, i=i: xs[i] if xs[i] else None)

    keep = [
        "task", "model_name", "model_id", "provider", "rep", "temperature",
        "cue", "score", "prompt", "response_clean", "ts_utc", "hash",
    ]
    if task == "dat":
        keep = keep[:8] + [f"noun_{i}" for i in range(10)] + keep[8:]
    return df[[c for c in keep if c in df.columns]]

for task in (
    "dat", 
    "aut",
    "cwt"):
    
    print(f"Parsing {task.upper()}...")
    task = task.lower()
    df = pd.DataFrame.from_dict(alp.load_pickles(task), orient="index")
    df = load_task(df,task)
    print(df.shape,df.columns)
    df.to_csv(f"./data/{task.upper()}_AI_2026.csv", index=False)
    print('Saved...')

Parsing DAT...


dat: 100%|████████████████████████████████████████████████████████| 10193/10193 [00:21<00:00, 480.03it/s]


(10193, 22) Index(['task', 'model_name', 'model_id', 'provider', 'rep', 'temperature',
       'cue', 'score', 'noun_0', 'noun_1', 'noun_2', 'noun_3', 'noun_4',
       'noun_5', 'noun_6', 'noun_7', 'noun_8', 'noun_9', 'prompt',
       'response_clean', 'ts_utc', 'hash'],
      dtype='object')
Saved...
Parsing AUT...


aut: 100%|████████████████████████████████████████████████████████| 17506/17506 [00:57<00:00, 303.35it/s]


(17506, 12) Index(['task', 'model_name', 'model_id', 'provider', 'rep', 'temperature',
       'cue', 'score', 'prompt', 'response_clean', 'ts_utc', 'hash'],
      dtype='object')
Saved...
Parsing CWT...


cwt: 100%|████████████████████████████████████████████████████████| 16278/16278 [00:39<00:00, 409.75it/s]


(16278, 12) Index(['task', 'model_name', 'model_id', 'provider', 'rep', 'temperature',
       'cue', 'score', 'prompt', 'response_clean', 'ts_utc', 'hash'],
      dtype='object')
Saved...


In [10]:
for task in ("dat", "aut", "cwt"):
    raw = pd.DataFrame.from_dict(alp.load_pickles(task), orient="index")
    print(raw.model_name.value_counts())
    print(raw.loc[raw.model_name.eq(m),"temperature"].fillna("default").value_counts())

dat: 100%|████████████████████████████████████████████████████████| 10193/10193 [00:21<00:00, 464.22it/s]


model_name
claude-opus-4.7      700
gpt-3.5-turbo        600
grok-4.3             600
grok-build-0.1       600
grok-4.5             600
llama-3.2-3b         450
gpt-4o-mini          450
llama-4-scout        450
claude-opus-5        450
llama-3.1-8b         450
claude-sonnet-4.5    450
claude-opus-4.5      450
llama-4-maverick     450
gpt-4-turbo          450
gpt-4o               450
claude-haiku-4.5     450
gpt-5.4              450
grok-4.2             400
grok-4.6             350
gpt-5.6-sol          350
deepseek-3.2         275
deepseek-2.5-chat    275
kimi-k2               20
deepseek-r1            5
gpt-5                  5
gpt-5-mini             5
grok-code-fast         5
qwen-turbo             3
Name: count, dtype: int64
Series([], Name: count, dtype: int64)


aut: 100%|████████████████████████████████████████████████████████| 17506/17506 [00:45<00:00, 381.84it/s]


model_name
claude-opus-4.7          1109
claude-opus-5            1105
gpt-3.5-turbo             921
gpt-4o                    870
grok-4.6                  862
gpt-5.4                   861
grok-4.2                  832
grok-build-0.1            831
claude-haiku-4.5          830
llama-3.1-8b              829
gpt-4o-mini               829
claude-opus-4.5           829
llama-4-scout             828
grok-4.5                  828
llama-3.2-3b              828
claude-sonnet-4.5         827
grok-4.3                  825
gpt-4-turbo               825
llama-4-maverick          823
deepseek-3.2              251
deepseek-2.5-chat         250
grok-4                    250
grok-code-fast             60
deepseek-r1                58
gpt-5.6-sol                42
kimi-k2                    41
deepseek-4-pro             40
deepseek-4-flash-0731      21
llama-3.3-70b               1
Name: count, dtype: int64
Series([], Name: count, dtype: int64)


cwt: 100%|███████████████████████████████████████████████████████| 16278/16278 [00:16<00:00, 1003.69it/s]


model_name
claude-opus-4.7      1120
claude-opus-5        1116
grok-build-0.1        856
gpt-5.6-sol           802
grok-4.6              797
claude-opus-4.5       749
claude-sonnet-4.5     744
grok-4.2              722
gpt-4o                719
gpt-3.5-turbo         719
grok-4.3              719
gpt-5.4               719
llama-4-maverick      717
gpt-4o-mini           717
llama-3.1-8b          717
llama-4-scout         716
claude-haiku-4.5      716
grok-4.5              715
gpt-4-turbo           713
llama-3.2-3b          710
deepseek-3.2          250
deepseek-2.5-chat     250
grok-4                250
grok-code-fast          5
gpt-5-mini              5
deepseek-r1             5
kimi-k2                 5
gpt-5                   5
Name: count, dtype: int64
Series([], Name: count, dtype: int64)
